In [ ]:
from pathlib import Path
import random

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import accelforge as af

In [ ]:
ARCH = Path("designs/arch.yaml")
WORKLOAD = Path("designs/gpt3_6.7B_kv_cache.yaml")
VALIDATOR = Path("designs/gpt3_175b_kv_cache.yaml")
MAPPING = Path("designs/gpt3_6.7B_kv_cache_mapping.yaml")

In [ ]:
MAPPER_JOBS = 1

lookaheads_to_test = list(range(4, 132, 4))
token_sweep = list(range(512, 4096 + 128, 128))
output_path = Path("milestone_2_results.csv")

def batch_size_for_lookahead(lookahead: int) -> int:
    # Use one query vector per speculative position under test.
    return max(1, int(lookahead))

af.set_n_parallel_jobs(MAPPER_JOBS)

In [ ]:
# ------------------------------------------------------------------
# Acceptance proxy
#
# We do not have a verified paper with the full GPT-3 6.7B token
# logprob histogram in a directly usable form, so we use an amortized
# acceptance-rate proxy instead.
#
# Source 1:
#   Yan et al., "Decoding Speculative Decoding", NAACL 2025
#   https://aclanthology.org/2025.naacl-long.328.pdf
#   Figure 2 shows OPT-6.7B draft TAR of about:
#     - 6.1 on MMLU
#     - 5.0 on Hellaswag
#   with a draft length gamma ~= 8.
#
# Source 2:
#   Leviathan et al., "Fast Inference from Transformers via
#   Speculative Decoding", ICML 2023
#   https://arxiv.org/abs/2211.17192
#
# We infer a token acceptance probability alpha by solving:
#   TAR ~= alpha + alpha^2 + ... + alpha^gamma
#
# Then, for each speculative token, we draw a random number and
# accept while random <= alpha, stopping at the first rejection.
# This is an inference from the papers, not a directly reported
# GPT-3 6.7B token histogram.
# ------------------------------------------------------------------
ACCEPTANCE_PROFILE = "mean"   # choose from: "mmlu", "hellaswag", "mean"
SOURCE_GAMMA = 8
TAR_BY_PROFILE = {
    "mmlu": 6.1,
    "hellaswag": 5.0,
}
TAR_BY_PROFILE["mean"] = sum(TAR_BY_PROFILE.values()) / len(TAR_BY_PROFILE)

BASE_RANDOM_SEED = 65930
ACCEPTANCE_MONTE_CARLO_TRIALS = 256


def matched_tokens_from_alpha(alpha: float, gamma: int) -> float:
    return sum(alpha ** i for i in range(1, gamma + 1))


def solve_alpha_from_tar(target_tar: float, gamma: int = SOURCE_GAMMA) -> float:
    lo, hi = 0.0, 0.999999
    for _ in range(80):
        mid = (lo + hi) / 2.0
        if matched_tokens_from_alpha(mid, gamma) < target_tar:
            lo = mid
        else:
            hi = mid
    return (lo + hi) / 2.0


ACCEPTANCE_PROB = solve_alpha_from_tar(
    TAR_BY_PROFILE[ACCEPTANCE_PROFILE],
    gamma=SOURCE_GAMMA,
)

print(
    {
        "arch": str(ARCH),
        "workload": str(WORKLOAD),
        "validator": str(VALIDATOR),
        "mapping": str(MAPPING),
        "batch_size_policy": "batch_size_for_lookahead(lookahead) = lookahead",
        "example_batch_sizes": {
            lookahead: batch_size_for_lookahead(lookahead)
            for lookahead in lookaheads_to_test[:4]
        },
        "acceptance_profile": ACCEPTANCE_PROFILE,
        "source_tar": TAR_BY_PROFILE[ACCEPTANCE_PROFILE],
        "inferred_acceptance_prob": ACCEPTANCE_PROB,
        "mc_trials": ACCEPTANCE_MONTE_CARLO_TRIALS,
    }
)


def make_spec(workload_path: Path, tokens: int, n_new_tokens: int, batch_size: int):
    return af.Spec.from_yaml(
        ARCH,
        workload_path,
        MAPPING,
        jinja_parse_data={
            "BATCH_SIZE": batch_size,
            "N_TOKENS": tokens,
            "N_NEW_TOKENS": n_new_tokens,
        },
    )
    

In [ ]:
def simulate_accepted_tokens(
    tokens: int,
    lookahead: int,
    accept_prob: float = ACCEPTANCE_PROB,
    trials: int = ACCEPTANCE_MONTE_CARLO_TRIALS,
    base_seed: int = BASE_RANDOM_SEED,
):
    accepted_counts = []

    for trial in range(trials):
        rng = random.Random(base_seed + tokens * 1000 + lookahead * 100 + trial)
        accepted = 0

        for _ in range(lookahead):
            if rng.random() <= accept_prob:
                accepted += 1
            else:
                break

        accepted_counts.append(accepted)

    accepted_counts = np.asarray(accepted_counts, dtype=float)
    emitted_counts = accepted_counts + 1.0

    return {
        "accepted_tokens_mean": float(accepted_counts.mean()),
        "accepted_tokens_std": float(accepted_counts.std(ddof=0)),
        "emitted_tokens_mean": float(emitted_counts.mean()),
        "emitted_tokens_std": float(emitted_counts.std(ddof=0)),
        "acceptance_rate_mean": float(accepted_counts.mean() / lookahead),
    }


def evaluate_workload(tokens: int, lookahead: int):
    batch_size = batch_size_for_lookahead(lookahead)
    total_energy = 0.0
    total_latency = 0.0

    # Draft model proposes `lookahead` single-token steps.
    for i in range(lookahead):
        draft_spec = make_spec(
            workload_path=WORKLOAD,
            tokens=tokens + i,
            n_new_tokens=1,
            batch_size=batch_size,
        )
        result = draft_spec.evaluate_mapping()
        total_energy += float(result.energy())
        total_latency += float(result.latency())

    # Validator verifies a block of size lookahead + 1.
    validator_spec = make_spec(
        workload_path=VALIDATOR,
        tokens=tokens,
        n_new_tokens=lookahead + 1,
        batch_size=batch_size,
    )
    result = validator_spec.evaluate_mapping()
    total_energy += float(result.energy())
    total_latency += float(result.latency())

    acceptance_stats = simulate_accepted_tokens(tokens=tokens, lookahead=lookahead)
    emitted_tokens_mean = acceptance_stats["emitted_tokens_mean"]
    average_energy_per_token = total_energy / emitted_tokens_mean
    average_latency_per_token = total_latency / emitted_tokens_mean

    return {
        "batch_size": batch_size,
        "energy": total_energy,
        "latency": total_latency,
        "energy_per_token": average_energy_per_token,
        "latency_per_token": average_latency_per_token,
        "Average Energy per token": average_energy_per_token,
        "Average latency per token": average_latency_per_token,
        **acceptance_stats,
    }


def default_kv_cached(tokens: int, lookahead: int):
    batch_size = batch_size_for_lookahead(lookahead)
    generated_tokens = lookahead + 1
    total_energy = 0.0
    total_latency = 0.0

    for i in range(generated_tokens):
        validator_spec = make_spec(
            workload_path=VALIDATOR,
            tokens=tokens + i,
            n_new_tokens=1,
            batch_size=batch_size,
        )
        result = validator_spec.evaluate_mapping()
        total_energy += float(result.energy())
        total_latency += float(result.latency())

    average_baseline_energy_per_token = total_energy / generated_tokens
    average_baseline_latency_per_token = total_latency / generated_tokens

    return {
        "baseline_generated_tokens": generated_tokens,
        "baseline_energy": total_energy,
        "baseline_latency": total_latency,
        "baseline_energy_per_token": average_baseline_energy_per_token,
        "baseline_latency_per_token": average_baseline_latency_per_token,
    }

In [ ]:
results_rows = []

for lookahead in lookaheads_to_test:
    for tokens in token_sweep:
        speculative = evaluate_workload(tokens=tokens, lookahead=lookahead)

        # Compare against target-only decoding of the same nominal block size.
        baseline = default_kv_cached(tokens=tokens, lookahead=lookahead)

        results_rows.append(
            {
                "tokens": tokens,
                "lookahead": lookahead,
                "batch_size": speculative["batch_size"],
                "acceptance_profile": ACCEPTANCE_PROFILE,
                "source_tar": TAR_BY_PROFILE[ACCEPTANCE_PROFILE],
                "acceptance_prob": ACCEPTANCE_PROB,
                "accepted_tokens_mean": speculative["accepted_tokens_mean"],
                "accepted_tokens_std": speculative["accepted_tokens_std"],
                "emitted_tokens_mean": speculative["emitted_tokens_mean"],
                "emitted_tokens_std": speculative["emitted_tokens_std"],
                "acceptance_rate_mean": speculative["acceptance_rate_mean"],
                "energy": speculative["energy"],
                "latency": speculative["latency"],
                "energy_per_token": speculative["energy_per_token"],
                "latency_per_token": speculative["latency_per_token"],
                "Average Energy per token": speculative["Average Energy per token"],
                "Average latency per token": speculative["Average latency per token"],
                "baseline_generated_tokens": baseline["baseline_generated_tokens"],
                "baseline_energy": baseline["baseline_energy"],
                "baseline_latency": baseline["baseline_latency"],
                "baseline_energy_per_token": baseline["baseline_energy_per_token"],
                "baseline_latency_per_token": baseline["baseline_latency_per_token"],
            }
        )

results_df = (
    pd.DataFrame(results_rows)
    .sort_values(["lookahead", "tokens"])
    .reset_index(drop=True)
)

results_df.to_csv(output_path, index=False)

print(f"Wrote {len(results_df)} rows to {output_path.resolve()}")
results_df.head()


In [ ]:
roundtrip_df = pd.read_csv(output_path)

expected_rows = len(lookaheads_to_test) * len(token_sweep)
expected_batch_sizes = {
    lookahead: batch_size_for_lookahead(lookahead)
    for lookahead in lookaheads_to_test
}

required_columns = [
    "tokens",
    "lookahead",
    "batch_size",
    "accepted_tokens_mean",
    "emitted_tokens_mean",
    "energy",
    "latency",
    "energy_per_token",
    "latency_per_token",
    "Average Energy per token",
    "Average latency per token",
    "baseline_energy",
    "baseline_latency",
    "baseline_energy_per_token",
    "baseline_latency_per_token",
]

assert len(roundtrip_df) == expected_rows, (len(roundtrip_df), expected_rows)
assert not roundtrip_df[required_columns].isna().any().any()
assert (roundtrip_df["energy"] > 0).all()
assert (roundtrip_df["latency"] > 0).all()
assert (roundtrip_df["Average Energy per token"] > 0).all()
assert (roundtrip_df["Average latency per token"] > 0).all()
assert (
    roundtrip_df["batch_size"]
    == roundtrip_df["lookahead"].map(expected_batch_sizes)
).all()
assert np.allclose(
    roundtrip_df["Average Energy per token"],
    roundtrip_df["energy"] / roundtrip_df["emitted_tokens_mean"],
)
assert np.allclose(
    roundtrip_df["Average latency per token"],
    roundtrip_df["latency"] / roundtrip_df["emitted_tokens_mean"],
)
assert np.allclose(
    roundtrip_df["Average Energy per token"],
    roundtrip_df["energy_per_token"],
)
assert np.allclose(
    roundtrip_df["Average latency per token"],
    roundtrip_df["latency_per_token"],
)

roundtrip_df.head(10)


In [ ]:
df = pd.read_csv(output_path).sort_values(["lookahead", "tokens"])
lookaheads = sorted(df["lookahead"].unique())

fig, axes = plt.subplots(2, 1, figsize=(10, 10), sharex=True)

for la in lookaheads:
    sub = df[df["lookahead"] == la]

    axes[0].plot(
        sub["tokens"],
        sub["energy"],
        marker="o",
        linewidth=2,
        label=f"Lookahead {int(la)}",
    )

    axes[1].plot(
        sub["tokens"],
        sub["latency"],
        marker="o",
        linewidth=2,
        label=f"Lookahead {int(la)}",
    )

axes[0].set_title("Energy vs Tokens")
axes[0].set_ylabel("Energy")
axes[0].grid(True, alpha=0.3)
axes[0].legend()

axes[1].set_title("Latency vs Tokens")
axes[1].set_xlabel("Tokens")
axes[1].set_ylabel("Latency")
axes[1].grid(True, alpha=0.3)
axes[1].legend()

plt.tight_layout()
plt.show()


In [ ]:
df = pd.read_csv(output_path).sort_values(["lookahead", "tokens"])
lookaheads = sorted(df["lookahead"].unique())

fig, axes = plt.subplots(2, 1, figsize=(10, 10), sharex=True)

for la in lookaheads:
    sub = df[df["lookahead"] == la]

    axes[0].plot(
        sub["tokens"],
        sub["Average Energy per token"],
        marker="o",
        linewidth=2,
        label=f"Lookahead {int(la)}",
    )

    axes[1].plot(
        sub["tokens"],
        sub["Average latency per token"],
        marker="o",
        linewidth=2,
        label=f"Lookahead {int(la)}",
    )

axes[0].set_title("Average Energy per Token vs Tokens")
axes[0].set_ylabel("Average Energy per Token")
axes[0].grid(True, alpha=0.3)
axes[0].legend()

axes[1].set_title("Average Latency per Token vs Tokens")
axes[1].set_xlabel("Tokens")
axes[1].set_ylabel("Average Latency per Token")
axes[1].grid(True, alpha=0.3)
axes[1].legend()

plt.tight_layout()
plt.show()
